In [1]:
# ============================================================
# Notebook : 04_proposed_model.ipynb
# Project  : Causal-Guided Multimodal Sensor Fusion for Elderly Risk Detection
# Purpose  : Train and evaluate the PROPOSED causal-guided graph fusion model
#            under household-level Group 5-fold cross-validation. For each test
#            fold the model uses that fold's causal graph (built from training
#            households only, in notebook 03). Preprocessing (standardization)
#            and class weighting are fit on training households only -> no leak.
#            Architecture: shared temporal conv per sensor node -> two masked
#            multi-head graph-attention layers with a per-head lag bias on the
#            causal edges -> mean pooling -> MLP classifier.
# Note     : Anonymous repository for double-blind review.
# ============================================================

from pathlib import Path
import numpy as np, pandas as pd, random
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
EPOCHS = 25; LR = 1e-3; WD = 1e-4; BATCH = 256

ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"; TAB_DIR = ROOT / "results" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

# ── Load windows and folds ───────────────────────────────────
z = np.load(PROC_DIR / "windows.npz", allow_pickle=True)
X = z["X"].astype(np.float32); y = z["y"].astype(np.int64); hh = z["hh"]
sample_fold = z["sample_fold"].astype(int)
FEAT = list(z["feat_cols"]); N = len(FEAT); WINDOW = int(z["window"])
N_FOLDS = int(sample_fold.max()) + 1
TAU_MAX = 6

def load_fold_graph(f):
    adj = np.load(PROC_DIR / f"causal_adj_fold{f}.npy")        # (N,N,tau+1) adj[i,j,t]=1 : i->j
    mask = np.zeros((N, N), np.float32); lag = np.zeros((N, N), np.float32)
    for i in range(N):
        for j in range(N):
            taus = [t for t in range(1, adj.shape[2]) if adj[i, j, t] == 1]
            if taus:
                mask[j, i] = 1.0; lag[j, i] = float(min(taus))   # [target, source]
    return mask, lag

# ── Proposed model ───────────────────────────────────────────
class ProposedModel(nn.Module):
    def __init__(self, mask, lag, use_lag=True, node_dim=32, heads=4, hid=64, dp=0.3):
        super().__init__()
        self.register_buffer("mask", torch.tensor(mask))
        self.register_buffer("lag", torch.tensor(lag))
        self.use_lag = use_lag; self.h = heads; self.hd = hid // heads; self.hid = hid
        self.temp = nn.Sequential(nn.Conv1d(1, 16, 3, padding=1), nn.ReLU(),
                                  nn.Conv1d(16, node_dim, 3, padding=1), nn.ReLU(),
                                  nn.AdaptiveAvgPool1d(1))
        self.q = nn.Linear(node_dim, hid); self.k = nn.Linear(node_dim, hid); self.v = nn.Linear(node_dim, hid)
        self.lagw = nn.Parameter(torch.zeros(self.h)) if use_lag else None
        self.proj = nn.Linear(hid, hid); self.drop = nn.Dropout(dp)
        self.clf = nn.Sequential(nn.Linear(hid, 32), nn.ReLU(), nn.Dropout(dp), nn.Linear(32, 2))
        self.n = mask.shape[0]
    def forward(self, x):
        B = x.shape[0]; Nn = self.n
        h = self.temp(x.reshape(B * Nn, 1, x.shape[-1])).squeeze(-1).view(B, Nn, -1)
        h = self.drop(h)
        q = self.q(h).view(B, Nn, self.h, self.hd).transpose(1, 2)
        k = self.k(h).view(B, Nn, self.h, self.hd).transpose(1, 2)
        v = self.v(h).view(B, Nn, self.h, self.hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / (self.hd ** 0.5)
        eye = torch.eye(Nn, device=x.device)[None, None]
        allow = ((self.mask[None, None] + eye) > 0).float()
        if self.use_lag:
            att = att + self.lagw.view(1, self.h, 1, 1) * self.lag[None, None]
        att = att.masked_fill(allow == 0, float("-inf"))
        att = torch.nan_to_num(torch.softmax(att, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, Nn, self.hid)
        out = self.drop(F.elu(self.proj(out)))
        return self.clf(out.mean(dim=1))

# ── Per-fold standardization (train households only) ─────────
def standardize(Xtr_raw, Xall_raw):
    mu = Xtr_raw.mean(axis=(0, 2), keepdims=True)
    sd = Xtr_raw.std(axis=(0, 2), keepdims=True); sd[sd == 0] = 1.0
    return (Xall_raw - mu) / sd

def loaders(Xs, idx_tr, idx_va, idx_te):
    def dl(idx, shuffle):
        ds = TensorDataset(torch.tensor(Xs[idx]), torch.tensor(y[idx]))
        return DataLoader(ds, batch_size=BATCH, shuffle=shuffle)
    return dl(idx_tr, True), dl(idx_va, False), dl(idx_te, False)

def train_eval(model, tr, va, te, cw):
    crit = nn.CrossEntropyLoss(weight=cw); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    best = -1; best_state = None
    for ep in range(EPOCHS):
        model.train()
        for xb, yb in tr:
            loss = crit(model(xb), yb); opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sch.step(); model.eval(); P = []; T = []
        with torch.no_grad():
            for xb, yb in va:
                P.extend(model(xb).argmax(1).numpy()); T.extend(yb.numpy())
        vf = f1_score(T, P, average="macro", zero_division=0)
        if vf > best: best = vf; best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    P = []; T = []; PR = []
    with torch.no_grad():
        for xb, yb in te:
            lo = model(xb); P.extend(lo.argmax(1).numpy()); T.extend(yb.numpy())
            PR.extend(F.softmax(lo, 1)[:, 1].numpy())
    return (f1_score(T, P, average="macro", zero_division=0),
            f1_score(T, P, average="binary", zero_division=0),
            roc_auc_score(T, PR), np.array(T), np.array(P), np.array(PR))

# ── Cross-validation ─────────────────────────────────────────
rows = []; pred_rows = []
for f in range(N_FOLDS):
    va = (f + 1) % N_FOLDS
    idx_te = np.where(sample_fold == f)[0]
    idx_va = np.where(sample_fold == va)[0]
    idx_tr = np.where((sample_fold != f) & (sample_fold != va))[0]
    Xs = standardize(X[idx_tr], X)
    cw = torch.tensor([1.0, (y[idx_tr] == 0).sum() / max((y[idx_tr] == 1).sum(), 1)], dtype=torch.float)
    mask, lag = load_fold_graph(f)
    torch.manual_seed(SEED)
    model = ProposedModel(mask, lag, use_lag=True)
    tr, vl, te = loaders(Xs, idx_tr, idx_va, idx_te)
    mf, bf, auc, T, P, PR = train_eval(model, tr, vl, te, cw)
    rows.append({"model": "Proposed (CausalGraph+Lag)", "fold": f,
                 "macro_f1": round(mf, 4), "binary_f1": round(bf, 4), "auc_roc": round(auc, 4)})
    for h, t, pr in zip(hh[idx_te], T, PR):
        pred_rows.append({"HH_ID": h, "fold": f, "y_true": int(t), "y_prob": float(pr)})
    print(f"fold {f}: macroF1={mf:.3f} binF1={bf:.3f} AUC={auc:.3f}")

res = pd.DataFrame(rows)
res.to_csv(TAB_DIR / "cv_proposed.csv", index=False)
pd.DataFrame(pred_rows).to_csv(TAB_DIR / "pred_proposed.csv", index=False)
m = res[["macro_f1", "binary_f1", "auc_roc"]].mean(); s = res[["macro_f1", "binary_f1", "auc_roc"]].std()
print("\nProposed model  (mean +/- std over %d folds):" % N_FOLDS)
print(f"  Macro F1 : {m.macro_f1:.3f} +/- {s.macro_f1:.3f}")
print(f"  Binary F1: {m.binary_f1:.3f} +/- {s.binary_f1:.3f}")
print(f"  AUC-ROC  : {m.auc_roc:.3f} +/- {s.auc_roc:.3f}")
print("Saved -> results/tables/cv_proposed.csv")
print("04_proposed_model complete.")


fold 0: macroF1=0.554 binF1=0.238 AUC=0.576


fold 1: macroF1=0.703 binF1=0.533 AUC=0.725


fold 2: macroF1=0.674 binF1=0.519 AUC=0.796


fold 3: macroF1=0.616 binF1=0.515 AUC=0.856


fold 4: macroF1=0.564 binF1=0.307 AUC=0.594

Proposed model  (mean +/- std over 5 folds):
  Macro F1 : 0.622 +/- 0.066
  Binary F1: 0.422 +/- 0.139
  AUC-ROC  : 0.709 +/- 0.123
Saved -> results/tables/cv_proposed.csv
04_proposed_model complete.
